# cdss_us_sbs - prepare the materials

A template, run in the private environment. The three extracts arrive **already split upstream**, so this notebook never splits: it shapes them, checks the split it was handed, and writes everything the pipeline and a discovery run read. Fill the `TODO`s top to bottom.

## What to prepare

Files, all written by this notebook to `data/cdss_us_sbs/`:

| File | Needed for | Made in |
| --- | --- | --- |
| `train.csv` · `valid.csv` · `test.csv` | every run - the tables the pipeline reads as given | step 5 |
| `column_descriptions.json` | discovery - what each column means | step 4, from your data dictionary |
| `column_mapping.csv` | a record of the column renames | step 2 |
| `few_shot.csv` | discovery - the example rows the proposer sees | step 6 |
| `screen_train.csv` · `screen_valid.csv` | discovery - the rows its screen fits and scores on | step 7 |

Keys in `configs/cdss_us_sbs.yaml`, filled in by you (step 3 prints what you need):

| Key | What to put there | Needed for |
| --- | --- | --- |
| `data.target` | the 0/1 outcome column | everything |
| `data.id_cols` | the account / date keys | the leakage check, the few-shot rows |
| `data.missing_values` | sentinel codes, e.g. `-9999` | cleaning |
| `features.base`, `features.new` (or `new_prefix`) | the incumbent set and the candidates | validation |
| `features.exclude` | non-numeric columns the model cannot use | validation |
| `discovery.task_description` | what a row is, the population, what the target means | discovery |
| `discovery.continuous_columns` (or `categorical_columns`) | which columns are measurements rather than codes | discovery |
| `discovery.screen_data` | `files` to screen on the samples from step 7, `splits` for the whole train / valid splits | discovery |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The repo root, wherever this notebook is run from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from preprocessing import build_sample, build_shot_batches, resolve_path, sanitize_columns, write_splits
from validation import load_config
from validation.data import resolve_features

pd.set_option("display.width", 160, "display.max_columns", 12)

## 1. Inputs

The three extracts, as `.csv` or `.parquet`. They are upstream of the pipeline, so they are named here rather than in the config.

In [ ]:
# TODO: the three extracts, split upstream
RAW = {
    "train": "TODO/path/to/train.csv",
    "valid": "TODO/path/to/valid.csv",
    "test":  "TODO/path/to/test.csv",
}
CONFIG = ROOT / "configs" / "cdss_us_sbs.yaml"

SEED = 42                          # the few-shot clustering
SHOTS, SHOT_BATCHES = 32, 10       # example rows per batch; one batch per discovery round
SCREEN_SIZE, SCREEN_BALANCE = {"train": 5000, "valid": 2000}, True   # TODO: the screen rows; balance when positives are rare

## 2. Load and sanitize

Column names become snake_case so they survive YAML, CSV and generated code; `mapping` keeps the originals. Every split must carry train's columns - a column train has and another split lacks stops here, and extra columns in valid / test are dropped.

In [ ]:
def read(path):
    path = Path(path)
    if path.suffix in (".parquet", ".pq"):
        return pd.read_parquet(path)
    return pd.read_csv(path, float_precision="round_trip")


frames, mapping = {}, None
for name, path in RAW.items():
    raw = read(path)
    frames[name], renamed = sanitize_columns(raw)
    mapping = renamed if name == "train" else mapping
    print(f"{name:<6} {raw.shape[0]:>9,} rows x {raw.shape[1]:>4} cols   {path}")

columns = list(frames["train"].columns)
for name in ("valid", "test"):
    missing = [c for c in columns if c not in frames[name].columns]
    assert not missing, f"{name!r} lacks column(s) that train has: {missing[:10]}"
    extra = [c for c in frames[name].columns if c not in columns]
    if extra:
        print(f"! {name!r} has {len(extra)} column(s) train lacks; dropping {extra[:5]}")
    frames[name] = frames[name][columns]

mapping[mapping["original"] != mapping["column"]].head(10)

## 3. Inspect - then fill in the config

What you need to fill in `configs/cdss_us_sbs.yaml`: which columns could be the 0/1 target, which are not numeric (encode them, or list them under `features.exclude`), and which carry the sentinel code.

In [ ]:
train = frames["train"]
profile = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(),
    "missing_rate": train.isna().mean().round(4),
    "n_-9999": (train == -9999).sum(),
})
binary = [c for c in columns if set(train[c].dropna().unique()) <= {0, 1}]
non_numeric = [c for c in columns if not pd.api.types.is_numeric_dtype(train[c])]
print(f"{len(columns)} columns; 0/1 columns (target candidates): {binary[:20]}")
print(f"non-numeric (encode, or list under features.exclude): {non_numeric[:20]}")
profile

Now edit `configs/cdss_us_sbs.yaml` - at least `data.target`, `data.id_cols`, `data.missing_values` and the candidates - and run the next cell. It reloads the config, so re-run it after every edit.

In [ ]:
cfg = load_config(CONFIG)
target, id_cols = cfg.data.target, list(cfg.data.id_cols)
assert target in columns, f"data.target {target!r} is not a column; pick one from the profile"
assert id_cols, "set data.id_cols - the leakage check and the few-shot rows need them"

for name, part in frames.items():
    print(f"{name:<6} n={len(part):>9,}  positive rate {part[target].mean():.4f}")
base, new = resolve_features(frames["train"], cfg.features, cfg.data)
print(f"{len(base)} incumbent feature(s), {len(new)} candidate(s): {new[:10]}")

## 4. Column descriptions

The file a discovery run reads to know what each column means - the most useful thing a proposer can be told. By default a column is described by its original header, which for an internal extract is usually a code, so write the meaning from the data dictionary. A column with no description reaches the proposer as a bare name.

In [ ]:
# TODO: what each column means, from the data dictionary. Keys are the sanitized
# names (see `mapping`); a column left out falls back to its original header.
DESCRIPTIONS = {
    # "util_pct": "Credit utilisation at the as-of date, percent of the limit.",
}

unknown = sorted(set(DESCRIPTIONS) - set(columns))
assert not unknown, f"DESCRIPTIONS names column(s) that are not in the table: {unknown}"
descriptions = {**dict(zip(mapping["column"], mapping["original"])), **DESCRIPTIONS}

undescribed = [c for c in base + new if c not in DESCRIPTIONS]
print(f"{len(base + new) - len(undescribed)} of {len(base + new)} features described; "
      f"{len(undescribed)} fall back to their header, e.g. {undescribed[:5]}")

## 5. Check against the config, then write

`write_splits` refuses to write anything if the tables disagree with the config: a candidate the config names but the extracts lack, a missing or non-binary target, splits with different columns, or **an id that appears in two splits** - leakage, which nothing downstream can detect.

In [ ]:
written = write_splits(cfg, frames, root=ROOT, mapping=mapping, descriptions=descriptions)
print(f"no id appears in two splits (checked on {id_cols})")
for name, path in written.items():
    print(f"{name:<13} {path.relative_to(ROOT)}")

## 6. Few-shot example rows

Only needed for discovery. The rows the proposer sees under every column, chosen per class by KMeans on train's incumbent columns - one row per cluster, 16 per class - so they cover the table and both classes appear even when positives are rare. Batch *r* is shown in round *r*. Sentinel codes count as missing for the clustering, as they will in the pipeline; the rows are saved as they appear in `train.csv`, plus a `batch` column.

In [ ]:
continuous = cfg.discovery.continuous_columns
categorical = ([c for c in base if c not in set(continuous)] if continuous is not None
               else [c for c in cfg.discovery.categorical_columns if c in base])

clean = frames["train"].copy()
if cfg.data.missing_values:
    clean[base] = clean[base].replace(list(cfg.data.missing_values), np.nan)
batches = build_shot_batches(clean, target, columns=base, categorical=categorical,
                             shots=SHOTS, batches=SHOT_BATCHES, seed=SEED)
# Saved as the raw rows - train.csv's own format, sentinels included - plus batch.
raw = frames["train"].set_index(id_cols[0], drop=False)
few_shot = pd.concat([raw.loc[b[id_cols[0]]].assign(batch=i) for i, b in enumerate(batches)],
                     ignore_index=True)
few_shot = few_shot[["batch", *frames["train"].columns]]

written["few_shot"] = resolve_path(cfg.discovery.few_shot_path, ROOT)
few_shot.to_csv(written["few_shot"], index=False)
print(f"{len(batches)} batches x {len(batches[0])} rows -> {written['few_shot'].relative_to(ROOT)}")
few_shot.groupby("batch")[target].agg(rows="size", positives="sum").T

## 7. Screen sample

Only needed for discovery with `discovery.screen_data: files`. The rows the screen fits each proposal on - a sample of **train** - and scores it on - a sample of **valid**: the cheap signal the proposer gets back every round, while the decision is made later, on the full splits. With `SCREEN_BALANCE` each class contributes up to half the rows, so rare positives are kept whole. Saved as the rows themselves, in the same format as `train.csv`.

Scoring on valid means the proposer's feedback comes from valid rows, so valid stops being an independent check on what it proposes; test still is. To keep valid independent, draw both samples from disjoint rows of train.

In [ ]:
screen = {
    part: build_sample(frames[part], target, SCREEN_SIZE[part],
                       balance=SCREEN_BALANCE, seed=SEED)
    for part in ("train", "valid")
}
for part, rows in screen.items():
    path = written[f"screen_{part}"] = resolve_path(cfg.discovery.screen_paths[part], ROOT)
    rows.to_csv(path, index=False)
    print(f"screen {part:<5} {len(rows):>7,} rows, {int(rows[target].sum()):,} positive "
          f"-> {path.relative_to(ROOT)}")

## Next

From the repo root:

```bash
autofe -c configs/cdss_us_sbs.yaml                                # validate the declared candidates
autofe -c configs/cdss_us_sbs.yaml --set discovery.enabled=true   # plus LLM-proposed ones
```

Discovery needs `discovery.task_description` written and an LLM credential (`OPENAI_API_KEY`, or `discovery.llm.backend: safechain`). `notebooks/usage.ipynb` walks through a full run on the bankruptcy demo.